# Group 7 — Fake News Detection: Setup & Data
**Owner:** Arwa Alqahtani (Team Leader / Data)

This part of the notebook:
1. loads the ISOT Fake News Dataset (`Fake.csv`, `True.csv`) and gives **Fake = 1, Real = 0**,
2. removes empty articles and duplicates **before** splitting,
3. builds the model input (**title + first 200 words of the body**) in two versions: `input` (original) and `input_clean` (source cues removed),
4. makes a stratified **70 / 15 / 15** train / dev / test split with a fixed seed and saves `train.csv`, `dev.csv`, `test.csv`,
5. checks that no example appears in more than one split.

The `subject` and `date` columns are **not used**: their values differ completely between the two classes, so they would leak the label.

## Setup

In [1]:
import os, re, sys, random
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split

SEED = 42                      # one seed for every random step in the project
random.seed(SEED)
np.random.seed(SEED)

DATA_DIR = Path("data")        # relative paths only (guidelines: reproducibility)
SPLIT_DIR = DATA_DIR / "splits"
DATA_DIR.mkdir(exist_ok=True)
SPLIT_DIR.mkdir(exist_ok=True)

N_WORDS = 200                  # body words kept in the model input (from the proposal)

print("Python      ", sys.version.split()[0])
print("pandas      ", pd.__version__)
print("numpy       ", np.__version__)
print("scikit-learn", sklearn.__version__)

Python       3.13.18
pandas       2.2.3
numpy        2.4.6
scikit-learn 1.6.1


## Data
### Load the dataset
The notebook finds the data in this order, so **Run all works from a fresh session**:
1. `data/Fake.csv` and `data/True.csv` already there → used directly;
2. otherwise a Kaggle `.zip` uploaded to the session (or to `data/`) → unzipped into `data/`;
3. otherwise downloaded from Kaggle with `kagglehub` (pre-installed on Colab; public datasets need no login).

The row counts are checked, so every run uses exactly the same files.
Note: the Kaggle data card says 23,502 fake articles, but `Fake.csv` actually has **23,481** rows; we report the real file counts.

In [2]:
import zipfile, shutil

KAGGLE_DATASET = "clmentbisaillon/fake-and-real-news-dataset"   # ISOT Fake.csv / True.csv on Kaggle
EXPECTED_ROWS = {"Fake.csv": 23481, "True.csv": 21417}          # row counts of the ISOT files

fake_path, true_path = DATA_DIR / "Fake.csv", DATA_DIR / "True.csv"

def have_data():
    return fake_path.exists() and true_path.exists()

# 2) a Kaggle zip uploaded by hand (e.g. "archive (13).zip")
if not have_data():
    for z in list(Path(".").glob("*.zip")) + list(DATA_DIR.glob("*.zip")):
        with zipfile.ZipFile(z) as zf:
            names = {Path(n).name: n for n in zf.namelist()}
            if "Fake.csv" in names and "True.csv" in names:
                for name in ["Fake.csv", "True.csv"]:
                    with zf.open(names[name]) as src_f, open(DATA_DIR / name, "wb") as dst_f:
                        shutil.copyfileobj(src_f, dst_f)
                print("Unzipped", z)
                break

# 3) download from Kaggle
if not have_data():
    import kagglehub
    src = Path(kagglehub.dataset_download(KAGGLE_DATASET))
    for name in ["Fake.csv", "True.csv"]:
        found = next(src.rglob(name))          # file may sit in a sub-folder
        shutil.copy(found, DATA_DIR / name)
    print("Downloaded from Kaggle:", KAGGLE_DATASET)

fake = pd.read_csv(fake_path)
real = pd.read_csv(true_path)
fake["label"] = 1      # Fake = 1
real["label"] = 0      # Real = 0

# a stable id that points back to the original file and row
fake["id"] = ["fake_" + str(i) for i in range(len(fake))]
real["id"] = ["real_" + str(i) for i in range(len(real))]

raw = pd.concat([fake, real], ignore_index=True)
raw = raw[["id", "title", "text", "label"]]       # subject and date dropped on purpose (label leakage)

assert len(fake) == EXPECTED_ROWS["Fake.csv"] and len(real) == EXPECTED_ROWS["True.csv"],     f"Unexpected files: Fake={len(fake)}, True={len(real)}"
print("Fake.csv rows:", len(fake))
print("True.csv rows:", len(real))
print("Total rows   :", len(raw))

Using Colab cache for faster access to the 'fake-and-real-news-dataset' dataset.
Downloaded from Kaggle: clmentbisaillon/fake-and-real-news-dataset
Fake.csv rows: 23481
True.csv rows: 21417
Total rows   : 44898


### Source-cue removal: `remove_cues`
In ISOT, real articles carry Reuters cues and fake articles carry their own artefacts.
A model could score highly by learning these cues instead of the content.
`remove_cues` removes:

| Cue | Mostly in | Example |
|---|---|---|
| Reuters dateline at the start | Real | `WASHINGTON (Reuters) - ` |
| The word *Reuters* anywhere else | Real | `told Reuters`, `Thomson Reuters` |
| URLs and Twitter picture links | Fake | `https://t.co/...`, `pic.twitter.com/...` |
| Twitter handles | Fake | `@realDonaldTrump` |
| Image-credit phrases | Fake | `Featured image via Getty Images`, `Photo by ...` |
| Reuters boilerplate | Real | `The following statements were posted to the verified Twitter accounts of ... Reuters has not edited the statements ...`, `(Reporting by ...; Editing by ...)` |
| Site names | Fake | `21st Century Wire`, `21WIRE` |
| Media tags in titles | Fake | `(VIDEO)`, `[TWEETS]`, `(WATCH)` |

The rules are fixed by hand (nothing is learned from the data), so applying them before the split does not leak test information.

In [3]:
_DATELINE   = re.compile(r"^[^\n]{0,150}?\(\s*reuters\s*\)\s*[-\u2013\u2014]*\s*", re.I)
_URL        = re.compile(r"(https?://\S+|www\.\S+|pic\.twitter\.com/\S+|\bbit\.ly/\S+|\bt\.co/\S+)", re.I)
_HANDLE     = re.compile(r"@\w+")
_IMG_CREDIT = re.compile(r"(featured image|header image|images?|photos?|screenshot|screen capture|screengrab)"
                         r"\s*(via|by|credit|credits|courtesy of|from)\b[^.\n]*", re.I)
_SITE_NAMES = re.compile(r"getty\s*images|\b(?:21st century wire|21wire|reuters)\b", re.I)
_MEDIA_TAG  = re.compile(r"[\(\[]\s*(video|videos|tweet|tweets|watch|image|images|photo|photos|audio|"
                         r"screenshot|screenshots|details|graphic video)\s*[\)\]]", re.I)
# Reuters note on "Trump on Twitter" articles; matched in 3 parts with \s+ so extra or
# non-breaking spaces between words do not stop the match
_TWEET_NOTE = [
    re.compile(r"the\s+following\s+statements?\s+(?:was|were)\s+posted\s+to\s+the\s+verified\s+"
               r"twitter\s+accounts?\s+of\s+(?:u\.\s*s\.\s+)?(?:president(?:-elect)?\s+)?donald\s+(?:j\.\s+)?trump"
               r"(?:\s*,?\s*(?:and\s+)?@\w+)*\s*\.?", re.I),
    re.compile(r"the\s+opinions?\s+expressed\s+(?:are|is)\s+(?:his|her|their)\s+own\s*\.?", re.I),
    re.compile(r"(?:reuters\s+)?has\s+not\s+edited\s+the\s+statements?\s+or\s+confirmed\s+(?:their|its)\s+accuracy\s*\.?", re.I),
]
_CREDITS    = re.compile(r"\(\s*(?:additional\s+)?(?:reporting|writing|editing|compiled|this story|this version)\b[^)]*\)", re.I)
_EMPTY_PAR  = re.compile(r"\(\s*\)|\[\s*\]")
_SPACES     = re.compile(r"\s+")

def remove_cues(text: str) -> str:
    """Remove source-identifying cues from a title or article body.
    Repeated until nothing changes, because removing one cue can bring two
    parts of another cue together, e.g. "Image (VIDEO) via YouTube"."""
    if not isinstance(text, str):
        return ""
    t = _DATELINE.sub("", text)
    for _ in range(5):
        before = t
        for rx in _TWEET_NOTE:
            t = rx.sub(" ", t)
        t = _CREDITS.sub(" ", t)
        t = _URL.sub(" ", t)
        t = _HANDLE.sub(" ", t)
        t = _IMG_CREDIT.sub(" ", t)
        t = _MEDIA_TAG.sub(" ", t)
        t = _SITE_NAMES.sub(" ", t)
        t = _EMPTY_PAR.sub(" ", t)
        t = _SPACES.sub(" ", t)
        if t == before:
            break
    return _SPACES.sub(" ", t).strip()

def first_words(text: str, n: int = N_WORDS) -> str:
    return " ".join(str(text).split()[:n])

def make_input(title: str, body: str) -> str:
    """Model input = title + first N_WORDS words of the body (same for both methods)."""
    return (str(title).strip() + ". " + first_words(body)).strip()

# quick checks that the rules do what we expect
tests = {
    "WASHINGTON (Reuters) - The Senate voted on Tuesday.": "The Senate voted on Tuesday.",
    "LONDON/PARIS (Reuters) -  Talks resumed.": "Talks resumed.",
    "He told Reuters the plan was final.": "He told the plan was final.",
    "Watch here https://t.co/abc123 pic.twitter.com/XyZ now": "Watch here now",
    "— Donald J. Trump (@realDonaldTrump) May 5, 2017": "— Donald J. Trump May 5, 2017",
    "The end. Featured image via Getty Images": "The end.",
    "Photo by Chip Somodevilla/Getty Images. Next sentence.": ". Next sentence.",
    "Trump SLAMS reporter (VIDEO)": "Trump SLAMS reporter",
    "21st Century Wire says the vote was rigged": "says the vote was rigged",
    ("The following statements were posted to the verified Twitter accounts of U.S. President Donald Trump, "
     "@realDonaldTrump and @POTUS. The opinions expressed are his own. Reuters has not edited the statements "
     "or confirmed their accuracy. @realDonaldTrump : - Tax cuts"): ": - Tax cuts",
    "Prices rose. (Reporting by Jane Doe; Editing by Tom Brown)": "Prices rose.",
    ("The following  statements were posted to the verified Twitter accounts of U.S. President Donald Trump, "
     "@realDonaldTrump and @POTUS.  The opinions expressed are his own. Reuters has not edited the  statements "
     "or confirmed their accuracy.  @realDonaldTrump : - Tax cuts"): ": - Tax cuts",
    "The Federal Reserve kept rates unchanged.": "The Federal Reserve kept rates unchanged.",
}
for raw_text, expected in tests.items():
    got = remove_cues(raw_text)
    assert got == expected, f"\n input:    {raw_text!r}\n expected: {expected!r}\n got:      {got!r}"
print(f"All {len(tests)} remove_cues checks passed.")

All 13 remove_cues checks passed.


### Build the model input and drop empty articles
`input` = title + first 200 words of the original body.
`input_clean` = cues removed from the title and the **full** body first, then the first 200 words are taken, so the clean version still has up to 200 words of content.

In [4]:
df = raw.copy()
df["title"] = df["title"].fillna("").astype(str)
df["text"]  = df["text"].fillna("").astype(str)

n0 = len(df)
empty_body = df["text"].str.strip() == ""
print("Rows with an empty body (dropped):", int(empty_body.sum()), "->", df.loc[empty_body, "label"].map({1: "Fake", 0: "Real"}).value_counts().to_dict())
df = df[~empty_body].copy()

df["input"]       = [make_input(t, b) for t, b in zip(df["title"], df["text"])]
df["clean_title"] = df["title"].map(remove_cues)
df["clean_body"]  = df["text"].map(remove_cues)
df["input_clean"] = [make_input(t, b) for t, b in zip(df["clean_title"], df["clean_body"])]

# an article whose body was only links / credits has nothing left to classify
empty_clean = df["clean_body"].str.strip() == ""
print("Rows whose body is empty after cue removal (dropped):", int(empty_clean.sum()))
df = df[~empty_clean].copy()
print("Rows kept:", len(df), "of", n0)

Rows with an empty body (dropped): 631 -> {'Fake': 630, 'Real': 1}
Rows whose body is empty after cue removal (dropped): 84
Rows kept: 44183 of 44898


### Remove duplicates before splitting
Duplicates are found on normalised text (lowercase, single spaces), in two passes:
1. same article body,
2. same **clean model input** — two articles that differ only in a cue (e.g. the dateline) or only after word 200 would give the model the same input, so only one is kept.

If the same text appears with **both** labels, all its copies are removed, because the label is unreliable.

In [5]:
def norm(s: str) -> str:
    return re.sub(r"\s+", " ", str(s).lower()).strip()

def dedup(frame: pd.DataFrame, key_col: str, step: str) -> pd.DataFrame:
    key = frame[key_col].map(norm)
    n_labels = frame.groupby(key)["label"].transform("nunique")
    conflict = n_labels > 1
    out = frame[~conflict]
    out = out[~key[~conflict].duplicated(keep="first")]
    print(f"{step}: removed {int(conflict.sum())} conflicting-label rows and "
          f"{len(frame) - int(conflict.sum()) - len(out)} duplicates -> {len(out)} rows")
    return out

before = df["label"].value_counts()
df = dedup(df, "text", "Pass 1 (same body)")
df = dedup(df, "input_clean", "Pass 2 (same clean input)")
after = df["label"].value_counts()

dedup_table = pd.DataFrame({"before": before, "after": after}).rename(index={1: "Fake (1)", 0: "Real (0)"})
dedup_table.loc["Total"] = dedup_table.sum()
dedup_table

Pass 1 (same body): removed 0 conflicting-label rows and 5599 duplicates -> 38584 rows
Pass 2 (same clean input): removed 0 conflicting-label rows and 6 duplicates -> 38578 rows


,before,after
label,,
Real (0),21416,21184
Fake (1),22767,17394
Total,44183,38578


### Stratified 70 / 15 / 15 split and save
Split once into train (70%) and a temporary 30%, then split the 30% in half into dev and test. Both splits are stratified by label and use `SEED`.

In [6]:
cols = ["id", "label", "title", "input", "input_clean"]
data = df[cols].reset_index(drop=True)

train, temp = train_test_split(data, test_size=0.30, stratify=data["label"], random_state=SEED)
dev, test   = train_test_split(temp, test_size=0.50, stratify=temp["label"], random_state=SEED)

for name, part in [("train", train), ("dev", dev), ("test", test)]:
    part.sort_values("id").to_csv(SPLIT_DIR / f"{name}.csv", index=False)

split_table = pd.DataFrame({
    name: {"Fake (1)": int((p.label == 1).sum()), "Real (0)": int((p.label == 0).sum()), "Total": len(p),
           "Share of data": f"{len(p) / len(data):.1%}", "Fake share": f"{p.label.mean():.1%}"}
    for name, p in [("train", train), ("dev", dev), ("test", test)]
}).T
print("Saved to", SPLIT_DIR.resolve())
split_table

Saved to /content/data/splits


,Fake (1),Real (0),Total,Share of data,Fake share
train,12176,14828,27004,70.0%,45.1%
dev,2609,3178,5787,15.0%,45.1%
test,2609,3178,5787,15.0%,45.1%


### Checks: no overlap between splits, and cues really removed

In [7]:
# 1) no example (by id or by clean input) appears in two splits
for a_name, a, b_name, b in [("train", train, "dev", dev), ("train", train, "test", test), ("dev", dev, "test", test)]:
    assert not set(a.id) & set(b.id), f"id overlap {a_name}/{b_name}"
    assert not set(a.input_clean.map(norm)) & set(b.input_clean.map(norm)), f"text overlap {a_name}/{b_name}"
print("No id or text overlap between train, dev and test.")

# 2) how often each cue appears before and after cleaning, per class
cue_patterns = {
    "Reuters":            r"\breuters\b",
    "URL / pic.twitter":  r"https?://|www\.|pic\.twitter\.com",
    "@handle":            r"@\w+",
    "Image credit":       r"(?:featured image|photo|image)\s*(?:via|by)|getty images",
    "Media tag (VIDEO)":  r"[\(\[]\s*(?:video|tweets?|watch)\s*[\)\]]",
    "Reuters boilerplate": r"verified twitter accounts? of|has not edited the statements|\((?:additional )?(?:reporting|editing) by",
}
rows = []
for cue, pat in cue_patterns.items():
    row = {"Cue": cue}
    for lab, lab_name in [(0, "Real"), (1, "Fake")]:
        part = data[data.label == lab]
        row[f"{lab_name} original"] = f"{part.input.str.contains(pat, case=False, regex=True).mean():.1%}"
        row[f"{lab_name} clean"]    = f"{part.input_clean.str.contains(pat, case=False, regex=True).mean():.1%}"
    rows.append(row)
cue_table = pd.DataFrame(rows).set_index("Cue")
left = {cue: int(data.input_clean.str.contains(pat, case=False, regex=True).sum()) for cue, pat in cue_patterns.items()}
assert sum(left.values()) == 0, f"Cues left after cleaning: {left}"
print("Every cue is at 0 in input_clean.")
cue_table      # % of articles in each class whose model input contains the cue

No id or text overlap between train, dev and test.
Every cue is at 0 in input_clean.


,Real original,Real clean,Fake original,Fake clean
Cue,,,,
Reuters,99.8%,0.0%,0.7%,0.0%
URL / pic.twitter,0.1%,0.0%,12.9%,0.0%
@handle,0.9%,0.0%,16.5%,0.0%
Image credit,0.0%,0.0%,0.6%,0.0%
Media tag (VIDEO),0.0%,0.0%,34.6%,0.0%
Reuters boilerplate,0.7%,0.0%,0.0%,0.0%


### Before / after examples

In [8]:
pd.set_option("display.max_colwidth", 300)
examples = pd.concat([data[data.label == 0].sample(2, random_state=SEED),
                      data[data.label == 1].sample(2, random_state=SEED)])
for _, r in examples.iterrows():
    print(f"[{r.id}]  label = {'Fake' if r.label == 1 else 'Real'}")
    print("  input      :", r.input[:250], "...")
    print("  input_clean:", r.input_clean[:250], "...\n")

[real_19173]  label = Real
  input      : Turkish, Iranian presidents discuss Iraqi Kurdish referendum: Erdogan's office. ISTANBUL (Reuters) - Turkish President Tayyip Erdogan and Iranian President Hassan Rouhani spoke by phone regarding a Iraqi Kurdish independence referendum and voiced con ...
  input_clean: Turkish, Iranian presidents discuss Iraqi Kurdish referendum: Erdogan's office. Turkish President Tayyip Erdogan and Iranian President Hassan Rouhani spoke by phone regarding a Iraqi Kurdish independence referendum and voiced concern that it will cau ...

[real_12761]  label = Real
  input      : Zimbabwe court postpones U.S. citizen's subversion case to January. HARARE (Reuters) - A Zimbabwean court has postponed to Jan. 4 the trial of a U.S. citizen accused of attempting to undermine the authority of former Zimbabwean president Robert Mugab ...
  input_clean: Zimbabwe court postpones U.S. citizen's subversion case to January. A Zimbabwean court has postponed to Jan. 4 the trial

In [9]:
!zip -r splits.zip data/splits
from google.colab import files
files.download("splits.zip")

  adding: data/splits/ (stored 0%)
  adding: data/splits/dev.csv (deflated 79%)
  adding: data/splits/train.csv (deflated 79%)
  adding: data/splits/test.csv (deflated 79%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### How Method 1 and Method 2 use these files
```python
train = pd.read_csv("data/splits/train.csv")
dev   = pd.read_csv("data/splits/dev.csv")
test  = pd.read_csv("data/splits/test.csv")
# train and tune on train["input"] / dev["input"]
# evaluate on test["input"] AND test["input_clean"]  (same test examples)
# final Try-it model: retrain on train["input_clean"]; apply remove_cues() to the user's text first
```